# V2 — Phase 2 — Build features V2 + audit

**Objectif.** Construire `data-lake/features/company_year_features_v2/` en remplaçant la jointure d'identité INSEE *snapshot* de V1 par la jointure *période-aware* sur `clean/company_identity_periodic/` produite en Phase 1. Réintroduire dans le jeu de features les quatre variables exclues par V1 (`activity_code`, `legal_category_code`, `employee_size_bracket`, `administrative_status_at_cutoff`) sans réintroduire la fuite temporelle.

**Pourquoi.** L'analyse SHAP de V1 (Phase D) montre que ces quatre variables auraient été les drivers #1, #3 et #4 du modèle. Les exclure était la correction *cheap* de V1 ; les réintégrer via une jointure période-aware est l'amélioration centrale de V2 (gain attendu : +5 à +10 pp d'AP).

**Critère de validation pour passer à la Phase 3.**

1. Audit anti-fuite à 100 % — pour chaque row, la période INSEE jointe a `period_start <= prediction_date < period_end`.
2. Taux de NaN sur `activity_code` V2 ≤ taux V1 (la jointure période-aware ne doit pas *perdre* de couverture par rapport au snapshot).
3. Counts BODACC / INPI / financiers identiques à V1 sur un échantillon de 1 000 SIRENs (sanity check de non-régression).


## 1. Setup


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/zribi1/pfein.git'
BRANCH = 'ml-v2'
REPO_DIR = '/content/pfein'
BACKEND_DIR = f'{REPO_DIR}/back_end'

DRIVE_ROOT = '/content/drive/MyDrive/PFE ML Data/pfe_data'
DATA_LAKE_DRIVE = f'{DRIVE_ROOT}/data-lake'
DUCKDB_TMP = '/content/pfein_duckdb_tmp'

PERIODIC_IDENTITY = f'{DATA_LAKE_DRIVE}/clean/company_identity_periodic'

DATA_LAKE_LOCAL = '/content/pfein_phase2_lake'
FEATURES_V2_LOCAL = f'{DATA_LAKE_LOCAL}/features/company_year_features_v2'
FEATURES_V2_DRIVE = f'{DATA_LAKE_DRIVE}/features/company_year_features_v2'
LABELS_V2_DRIVE = f'{DATA_LAKE_DRIVE}/features/risk_labels_v2'

FEATURES_V1 = f'{DATA_LAKE_DRIVE}/features/company_year_features'

Path(DUCKDB_TMP).mkdir(parents=True, exist_ok=True)
Path(DATA_LAKE_LOCAL).mkdir(parents=True, exist_ok=True)

print('BACKEND_DIR        =', BACKEND_DIR)
print('PERIODIC_IDENTITY  =', PERIODIC_IDENTITY)
print('FEATURES_V2_LOCAL  =', FEATURES_V2_LOCAL)
print('FEATURES_V2_DRIVE  =', FEATURES_V2_DRIVE)


In [ ]:
import os, subprocess

if not Path(REPO_DIR).exists():
    !git clone --branch "$BRANCH" "$REPO_URL" "$REPO_DIR"

%cd $REPO_DIR
!git fetch origin
!git switch "$BRANCH" || git switch -c "$BRANCH" "origin/$BRANCH"
!git pull --ff-only origin "$BRANCH"
%cd $BACKEND_DIR

os.environ['DUCKDB_TEMP_DIRECTORY'] = DUCKDB_TMP
!pip install -q -r collabs/requirements-colab.txt


## 2. Vérifier que la Phase 1 (identité périodique) est en place


In [ ]:
import duckdb

p = Path(PERIODIC_IDENTITY)
files = list(p.rglob('*.parquet'))
if not files:
    raise SystemExit(
        f"Aucun parquet trouvé sous {p}.\n"
        f"Lancez d'abord la Phase 1 (v2_phase_1_identity_periodic.ipynb)."
    )
print(f'Parquet identité périodique : {len(files)} fichier(s)')
for f in files:
    print(f'  - {f.name}  ({f.stat().st_size / 1e9:.2f} GB)')

con = duckdb.connect()
schema = con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{PERIODIC_IDENTITY}/**/*.parquet') LIMIT 0"
).df()
n_rows = con.execute(
    f"SELECT COUNT(*) FROM read_parquet('{PERIODIC_IDENTITY}/**/*.parquet')"
).fetchone()[0]
con.close()
print(f'\nRows : {n_rows:,}')
print('Schema :')
print(schema[['column_name', 'column_type']].to_string(index=False))


## 3. Symlink les sources V1 partagées dans le data-lake local

Le builder V2 attend une arborescence `data-lake/clean/<table>/` complète. La couche `clean/company_identity_periodic/` vient de la Phase 1 (sur Drive). Les autres `clean/legal_events`, `clean/formalities_events`, `clean/annual_accounts`, `clean/financials` sont les tables V1 *réutilisées sans modification* (per roadmap). On les pointe via symlinks pour éviter une copie.

Le builder écrit sous `data-lake/features/company_year_features_v2/` — on stage sur SSD local pour éviter l'OOM FUSE.


In [ ]:
import os

clean_local = Path(DATA_LAKE_LOCAL) / 'clean'
clean_local.mkdir(parents=True, exist_ok=True)

shared_tables = [
    'company_identity_periodic',
    'legal_events',
    'formalities_events',
    'annual_accounts',
    'financials',
]
for t in shared_tables:
    src = Path(DATA_LAKE_DRIVE) / 'clean' / t
    dst = clean_local / t
    if dst.is_symlink() or dst.exists():
        if dst.is_symlink():
            dst.unlink()
        else:
            import shutil as _sh; _sh.rmtree(dst)
    if not src.exists():
        print(f'⚠️  Missing source on Drive: {src}')
        continue
    os.symlink(src, dst)
    print(f'symlinked {dst} -> {src}')

(Path(DATA_LAKE_LOCAL) / 'features').mkdir(parents=True, exist_ok=True)
print('\nLocal data-lake tree:')
!ls -la "$DATA_LAKE_LOCAL"
!ls -la "$DATA_LAKE_LOCAL/clean"


## 4. Lancer le build des features V2

**Args clés.**

- `--start-year 2017 --end-year 2024` : fenêtre standard (V1 utilise la même).
- `--year-batch-size 1` : on écrit année par année pour limiter le pic mémoire DuckDB.
- `--max-companies` : à passer pour un *smoke test* rapide (e.g. 200000), à retirer pour le run complet.

**Durée estimée.** Sur 52 GB Colab High-RAM, le build full prend 15-30 min selon le débit Drive (lectures `clean/*`). En smoke test à 200 K SIRENs, ~3-5 min.


In [ ]:
import shlex, subprocess, sys, time

SMOKE_TEST = True
MAX_COMPANIES = 200_000

cmd = [
    sys.executable, '-u',
    '-m', 'app.tools.v2.build_company_year_features_v2',
    '--data-lake-dir', DATA_LAKE_LOCAL,
    '--start-year', '2017',
    '--end-year',   '2024',
    '--year-batch-size', '1',
    '--overwrite',
]
if SMOKE_TEST:
    cmd += ['--max-companies', str(MAX_COMPANIES)]

print(' '.join(shlex.quote(p) for p in cmd))
print()
start = time.time()
result = subprocess.run(cmd, capture_output=True, text=True)
elapsed = time.time() - start

print(result.stdout[-4000:])
if result.returncode != 0:
    print('STDERR (tail):')
    print(result.stderr[-4000:])
    raise SystemExit(f'Build failed with code {result.returncode}')
print(f'\nDurée du build : {elapsed/60:.1f} min')


## 5. Sync local → Drive


In [ ]:
import shutil, json as _json

for local_sub, drive_target in [
    ('features/company_year_features_v2', FEATURES_V2_DRIVE),
    ('features/risk_labels_v2',           LABELS_V2_DRIVE),
]:
    src_dir = Path(DATA_LAKE_LOCAL) / local_sub
    dst_dir = Path(drive_target)
    if not src_dir.exists():
        print(f'⚠️  Skip {src_dir} — not built')
        continue
    dst_dir.mkdir(parents=True, exist_ok=True)
    moved = 0
    for f in src_dir.rglob('*'):
        if not f.is_file():
            continue
        rel = f.relative_to(src_dir)
        target = dst_dir / rel
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists():
            target.unlink()
        shutil.move(str(f), str(target))
        moved += 1
    print(f'Moved {moved} files: {src_dir} -> {dst_dir}')
    shutil.rmtree(src_dir, ignore_errors=True)

print('\nDrive contents:')
!ls -la "$FEATURES_V2_DRIVE"


## 6. Audit 1 — Anti-fuite (jointure période-aware)

Pour chaque row où `activity_code IS NOT NULL`, on vérifie qu'il existe une période INSEE telle que `period_start <= prediction_date < period_end`. Le builder le garantit par construction de la jointure SQL, mais on le confirme empiriquement.


In [ ]:
con = duckdb.connect()
con.execute(f"PRAGMA threads={os.cpu_count() or 4}")
con.execute("PRAGMA memory_limit='25GB'")

audit = con.execute(f"""
    WITH f AS (
        SELECT siren, prediction_date, activity_code
        FROM read_parquet('{FEATURES_V2_DRIVE}/**/*.parquet')
        WHERE activity_code IS NOT NULL
    ),
    i AS (
        SELECT siren, period_start, period_end, activity_code AS p_activity_code
        FROM read_parquet('{PERIODIC_IDENTITY}/**/*.parquet')
    ),
    joined AS (
        SELECT
            f.siren,
            f.prediction_date,
            f.activity_code,
            i.p_activity_code,
            i.period_start,
            i.period_end
        FROM f
        LEFT JOIN i
          ON i.siren = f.siren
         AND i.period_start <= f.prediction_date
         AND i.period_end   >  f.prediction_date
    )
    SELECT
        COUNT(*) AS rows_with_activity,
        SUM(CASE WHEN p_activity_code IS NULL THEN 1 ELSE 0 END) AS rows_no_matching_period,
        SUM(CASE WHEN p_activity_code IS NOT NULL
                  AND p_activity_code = activity_code
                 THEN 1 ELSE 0 END) AS rows_value_matches
    FROM joined
""").fetchone()

rows_total, rows_no_match, rows_value_ok = audit
print(f'Rows avec activity_code non-null         : {rows_total:,}')
print(f'Rows sans période INSEE correspondante   : {rows_no_match:,}  ({rows_no_match/max(rows_total,1):.4%})')
print(f'Rows dont la valeur jointe = celle stockée: {rows_value_ok:,}  ({rows_value_ok/max(rows_total,1):.4%})')

leak_free = rows_no_match == 0
value_consistent = rows_value_ok == rows_total
print()
print('✅ ANTI-FUITE OK' if leak_free and value_consistent else '❌ ANTI-FUITE KO')


## 6b. Diagnostic — distinguer *jointure ratée* vs *jointure ok mais valeur NULL*

Si l'audit 2 montre un taux de NaN élevé (>20 %) sur les colonnes INSEE, ce n'est pas forcément un bug de jointure. Pour un SIREN ancien dormant, INSEE peut avoir une seule période historique sans `activite_principale` ni `categorie_juridique` renseignés. La jointure réussit alors, mais les valeurs sont NULL.

Cette cellule décompose le taux de NaN en deux causes.


In [ ]:
diag = con.execute(f"""
    SELECT
        COUNT(*)                                                AS rows_total,
        SUM(CASE WHEN i.siren IS NULL THEN 1 ELSE 0 END)        AS rows_no_period_match,
        SUM(CASE WHEN i.siren IS NOT NULL
                  AND i.activity_code IS NULL
                 THEN 1 ELSE 0 END)                            AS rows_joined_but_null,
        SUM(CASE WHEN i.activity_code IS NOT NULL THEN 1 ELSE 0 END)
                                                                AS rows_joined_with_value
    FROM read_parquet('{FEATURES_V2_DRIVE}/**/*.parquet') f
    LEFT JOIN read_parquet('{PERIODIC_IDENTITY}/**/*.parquet') i
      ON i.siren = f.siren
     AND i.period_start <= f.prediction_date
     AND i.period_end   >  f.prediction_date
""").fetchone()

rt, no_match, joined_null, joined_ok = diag
print(f'Rows total                                    : {rt:,}')
print(f'  ├─ aucune période INSEE ne matche           : {no_match:,}  ({no_match/max(rt,1):.2%})')
print(f'  ├─ période trouvée mais activity_code NULL  : {joined_null:,}  ({joined_null/max(rt,1):.2%})')
print(f'  └─ période trouvée + activity_code renseigné: {joined_ok:,}  ({joined_ok/max(rt,1):.2%})')

if no_match > 0.1 * rt:
    print('
❌ Plus de 10% des rows n'ont aucune période matchante — bug de jointure ou sample biaisé.')
elif joined_null > 0.5 * rt:
    print('
⚠️  La jointure réussit mais les valeurs INSEE sont majoritairement NULL.')
    print('   Probable cause : le smoke-test échantillonne les SIRENs alphabétiquement (LIMIT 200000)')
    print('   et tombe sur des entités très anciennes / dormantes avec un historique INSEE pauvre.')
    print('   Re-runner avec SMOKE_TEST = False pour avoir une vraie mesure.')
else:
    print('
✅ Couverture INSEE saine.')


## 7. Audit 2 — Couverture INSEE (taux de NaN par colonne)


In [ ]:
cols_check = ['activity_code', 'legal_category_code', 'administrative_status_at_cutoff', 'employee_size_bracket']

stats = con.execute(f"""
    SELECT
        COUNT(*) AS rows_total,
        {", ".join(f"SUM(CASE WHEN {c} IS NULL THEN 1 ELSE 0 END) AS nan_{c}" for c in cols_check)}
    FROM read_parquet('{FEATURES_V2_DRIVE}/**/*.parquet')
""").fetchone()

rows_total = stats[0]
print(f'Rows total : {rows_total:,}\n')
header_col = "Colonne"
header_nan = "NaN"
header_pct = "% NaN"
print(f'{header_col:<40} {header_nan:>12} {header_pct:>8}')
print('-' * 64)
for i, c in enumerate(cols_check, start=1):
    n = stats[i]
    print(f'{c:<40} {n:>12,} {n/max(rows_total,1):>7.2%}')


## 8. Audit 3 — Non-régression sur les features non-identité

Sur un échantillon de 1 000 SIRENs aléatoires, vérifier que les colonnes BODACC (`legal_events_count_*`), INPI (`formalities_count_*`) et financières (`previous_year_revenue`, etc.) sont **identiques** à celles produites par V1. Toute divergence signale un bug introduit par le fork V1→V2.


In [ ]:
sample_sirens = con.execute(f"""
    SELECT DISTINCT siren
    FROM read_parquet('{FEATURES_V2_DRIVE}/**/*.parquet')
    USING SAMPLE 1000 ROWS (RESERVOIR, 42)
""").df()['siren'].tolist()
in_clause = ", ".join(f"'{s}'" for s in sample_sirens)

cols_check_reg = [
    'legal_events_count_all', 'legal_events_count_12m',
    'legal_risk_events_count_all', 'legal_risk_events_count_12m',
    'legal_distress_events_count_all', 'radiation_events_count_all',
    'formalities_count_all', 'formalities_count_12m',
    'cessation_formalities_count_all',
    'latest_revenue', 'latest_net_result',
    'revenue_growth_1y', 'net_result_change_1y',
]

v2 = con.execute(f"""
    SELECT siren, prediction_year, {", ".join(cols_check_reg)}
    FROM read_parquet('{FEATURES_V2_DRIVE}/**/*.parquet')
    WHERE siren IN ({in_clause})
    ORDER BY siren, prediction_year
""").df()

try:
    v1 = con.execute(f"""
        SELECT siren, prediction_year, {", ".join(cols_check_reg)}
        FROM read_parquet('{FEATURES_V1}/**/*.parquet')
        WHERE siren IN ({in_clause})
        ORDER BY siren, prediction_year
    """).df()
except Exception as e:
    print(f'⚠️  V1 features not available for comparison: {e}')
    v1 = None

if v1 is not None and len(v1):
    merged = v2.merge(v1, on=['siren', 'prediction_year'], suffixes=('_v2', '_v1'), how='inner')
    print(f'Lignes appariées : {len(merged):,}')
    hdr1 = "Colonne"
    hdr2 = "Δ ≠ 0"
    hdr3 = "%"
    print(f'\n{hdr1:<40} {hdr2:>10} {hdr3:>8}')
    print('-' * 60)
    for c in cols_check_reg:
        diff = (merged[f'{c}_v2'] != merged[f'{c}_v1']).fillna(False)
        both_nan = merged[f'{c}_v2'].isna() & merged[f'{c}_v1'].isna()
        diff = diff & ~both_nan
        n_diff = int(diff.sum())
        print(f'{c:<40} {n_diff:>10,} {n_diff/max(len(merged),1):>7.2%}')
else:
    print('⚠️  Comparaison V1/V2 sautée — V1 features absentes ou vides.')

con.close()


## 9. Mettre à jour le journal de phase

Une fois les trois audits validés (anti-fuite à 100 %, NaN raisonnables, non-régression à 0), ouvrir `docs/v2/v2_phase_log.md` et remplir la section Phase 2 avec :

- `Lignes features V2` (depuis le `_manifest.json`)
- `Joint rate INSEE` (1 − % NaN sur `activity_code`)
- `% NaN sur activity_code V2 vs V1`
- `Audit anti-fuite : passé / échoué`

Puis **passer à la Phase 3** (baseline iteratif sur 100 K rows).
